# 抖音热榜 Top 50：Scrapling

使用 Scrapling 的 `StealthyFetcher` 渲染抖音热榜页面，提取前 50 个关键词并保存为 CSV。

In [3]:
# 导入正则、CSV、时间和路径工具，用于解析字段和写入结果文件。
import csv
import re
from datetime import datetime, timezone
from pathlib import Path

# 导入 Scrapling 的浏览器抓取器，用于渲染抖音这个 JavaScript 页面。
from scrapling.fetchers import StealthyFetcher

# 设置目标页面和输出文件位置，路径相对于项目根目录。
HOT_URL = "https://www.douyin.com/hot"
OUTPUT_PATH = Path("collect/hot_scrapling_notebook.csv")

# 这些 CSS 选择器对应当前热榜页面中的条目、标题和热度节点。
ITEM_SELECTOR = "div.qe4SKjIp"
TITLE_SELECTOR = "div.OQtNjDJ9 a h3::text"
HEAT_SELECTOR = "div.m73YyvwD span.ZZFBf5Wm::text"
HEAT_BLOCK_SELECTOR = "div.m73YyvwD"

# 定义 CSV 的固定列顺序，便于后续管线继续读取。
CSV_FIELDS = ["rank", "title", "hot_url", "hot_id", "hot_value_k", "fetched_at"]

In [4]:
def parse_heat(text):
    # 从热度文本中提取数值和单位，例如 1123.7万 或 1234.5K。
    match = re.fullmatch(r"\s*([\d.]+)\s*(万|K|k)?\s*", text or "")
    if not match:
        # 页面格式异常时返回空值，不让单条数据中断整个采集。
        return None

    # 将匹配到的数字转换为浮点数，统一用于 CSV 输出。
    value = float(match.group(1))
    unit = match.group(2)

    # 把“万”转换成 K：1 万 = 10 K。
    if unit == "万":
        value *= 10

    # 返回统一为 K 单位的热度值。
    return value

In [6]:
# 启动无头 Chromium，等待页面完成 JavaScript 渲染和网络请求。
page = await StealthyFetcher.async_fetch(
    HOT_URL,
    network_idle=True,
    timeout=60000,
    wait=2000,
)

# 读取页面中的全部榜单条目；当前页面通常包含 51 项，其中 1 项是置顶内容。
items = page.css(ITEM_SELECTOR)
print(f"页面抓到条目数：{len(items)}")

[2026-10-20 04:45:55] INFO: Fetched (200) <GET https://www.douyin.com/hot> (referer: https://www.google.com/)


页面抓到条目数：51


In [7]:
# 创建结果列表，后面只保留前 50 条真正的热榜数据。
rows = []

# 逐条解析页面中的榜单容器。
for item in items:
    # 置顶项没有热度容器，因此跳过它，避免占用 Top 50 名额。
    if not item.css(HEAT_BLOCK_SELECTOR):
        continue

    # 从标题节点读取热榜关键词。
    title = item.css(TITLE_SELECTOR).get()

    # 读取详情链接，链接中同时包含热榜 ID 和标题 slug。
    href = item.css("a::attr(href)").get()

    # 读取原始热度文本，并转换成统一的 K 单位。
    heat_text = item.css(HEAT_SELECTOR).get()
    hot_value_k = parse_heat(heat_text) if heat_text else None

    # 将相对链接转换成完整的抖音链接。
    hot_url = page.urljoin(href) if href else ""

    # 从链接路径中提取数字热榜 ID。
    hot_id_match = re.search(r"/hot/(\d+)", href or "")
    hot_id = hot_id_match.group(1) if hot_id_match else ""

    # 用过滤置顶项后的顺序生成 1 到 50 的榜单排名。
    rows.append({
        "rank": len(rows) + 1,
        "title": (title or "").strip(),
        "hot_url": hot_url,
        "hot_id": hot_id,
        "hot_value_k": hot_value_k,
        "fetched_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    })

    # 达到 Top 50 后停止遍历，避免保存额外条目。
    if len(rows) >= 50:
        break

# 输出实际提取数量，方便快速确认页面结构是否正常。
print(f"提取关键词数：{len(rows)}")

提取关键词数：50


In [8]:
# 创建输出目录，目录不存在时自动创建。
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

# 将 Top 50 结果写入 UTF-8 CSV，确保中文标题正常保存。
with OUTPUT_PATH.open("w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=CSV_FIELDS)
    writer.writeheader()
    writer.writerows(rows)

# 打印输出位置和前 5 条结果，便于在 notebook 中立即检查。
print(f"已写入：{OUTPUT_PATH}")
for row in rows[:5]:
    print(f"#{row['rank']:>2} {row['title']} ({row['hot_value_k']}K热度)")

已写入：collect\hot_scrapling_notebook.csv
# 1 中国队夺拜仁青年杯冠军 (11421.0K热度)
# 2 嫁金钗晋江式演技模仿 (11315.0K热度)
# 3 何以中国当惊世界殊 (11102.0K热度)
# 4 S16瑞士轮首轮抽签结果 (11101.0K热度)
# 5 南北吃蟹大不同 (10545.0K热度)


In [9]:
# 检查是否正好得到 50 条，并确认关键字段没有为空。
assert len(rows) == 50, f"期望 50 条，实际得到 {len(rows)} 条"
assert [row["rank"] for row in rows] == list(range(1, 51))
assert all(row["title"] for row in rows)
assert all(row["hot_id"].isdigit() for row in rows)

# 验证通过后打印确认信息。
print("Top 50 校验通过")

Top 50 校验通过


## Videos under each

In [ ]:
# 在这里输入 Cookie：单个字符串或账号池都行。
# 示例：COOKIE_POOL = ["sessionid=...; sid_guard=...; ..."]
# 多个账号时按热榜关键词轮流使用；留空则自动读取 .env 的 DOUYIN_HOT_COOKIE。
COOKIE_POOL = [
    # "sessionid=xxx; ...",
]


In [ ]:
import json
import re
import urllib.parse
from datetime import datetime
from pathlib import Path

from scrapling.fetchers import AsyncStealthySession

VIDEO_OUTPUT_PATH = Path("collect/hot_scrapling_videos.csv")
SEARCH_URL = "https://www.douyin.com/search/{kw}?type=video"
SEARCH_API_RE = re.compile(r"aweme/v1/web/(general/)?search/single")
CAPTCHA_TEXT = "验证码"

VIDEO_FIELDS = [
    "keyword",
    "keyword_rank",
    "aweme_id",
    "desc",
    "title",
    "author_nickname",
    "author_sec_uid",
    "digg_count",
    "comment_count",
    "view_count",
    "share_count",
    "share_url",
]


def _load_cookie_from_env():
    env_path = Path(".env")
    if not env_path.exists():
        return ""
    for line in env_path.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if line.startswith("DOUYIN_HOT_COOKIE=") and len(line) > len("DOUYIN_HOT_COOKIE="):
            return line[len("DOUYIN_HOT_COOKIE="):].strip()
    return ""


def _parse_cookies(raw):
    pairs = [p.split("=", 1) for p in raw.split(";") if "=" in p]
    cookies = []
    for name, value in pairs:
        name, value = name.strip(), value.strip()
        if name and value:
            cookies.append({"name": name, "value": value, "domain": ".douyin.com", "path": "/"})
    return cookies


def _get_path(obj, path, default=None):
    cur = obj
    for part in path.split("."):
        if isinstance(cur, dict):
            cur = cur.get(part)
        elif isinstance(cur, list) and part.isdigit() and int(part) < len(cur):
            cur = cur[int(part)]
        else:
            return default
        if cur is None:
            return default
    return cur


def _first(obj, *paths, default=None):
    for p in paths:
        v = _get_path(obj, p)
        if v not in (None, "", [], {}):
            return v
    return default


def _num(v, default=0):
    if v is None or v == "":
        return default
    if isinstance(v, bool):
        return default
    if isinstance(v, (int, float)):
        return v
    if isinstance(v, str):
        s = v.replace(",", "").strip()
        try:
            return float(s) if "." in s else int(s)
        except ValueError:
            return default
    return default


def _walk_aweme(obj, out):
    if isinstance(obj, dict):
        if isinstance(obj.get("aweme_id"), str) and "desc" in obj:
            out.append(obj)
        for v in obj.values():
            _walk_aweme(v, out)
    elif isinstance(obj, list):
        for v in obj:
            _walk_aweme(v, out)


def _rsc_aweme_objects(text):
    found = []
    for m in re.finditer(r'self\.__pace_f\.push\(\[(\d+),\s*("(?:[^"\\]|\\.)*")\]\)', text):
        try:
            seg = json.loads(m.group(2))
        except Exception:
            continue
        if not isinstance(seg, str):
            continue
        starts = [seg.find("{"), seg.find("[")]
        starts = [i for i in starts if i != -1]
        if not starts:
            continue
        start = min(starts)
        try:
            data = json.loads(seg[start:])
        except Exception:
            continue
        _walk_aweme(data, found)
    seen, out = set(), []
    for d in found:
        aid = d.get("aweme_id")
        if aid in seen:
            continue
        seen.add(aid)
        out.append(d)
    return out


def _extract_video(item, keyword, keyword_rank):
    info = _first(item, "aweme_info", "aweme", default=item)
    if not isinstance(info, dict):
        return None
    video = info.get("video") or {}
    stats = info.get("statistics") or {}
    author = info.get("author") or {}
    return {
        "keyword": keyword,
        "keyword_rank": keyword_rank,
        "aweme_id": info.get("aweme_id", ""),
        "desc": (info.get("desc") or "").strip(),
        "title": (info.get("desc") or "").strip(),
        "author_nickname": author.get("nickname", ""),
        "author_sec_uid": author.get("sec_uid", ""),
        "digg_count": _num(stats.get("digg_count")),
        "comment_count": _num(stats.get("comment_count")),
        "view_count": _num(stats.get("play_count") or stats.get("view_count")),
        "share_count": _num(stats.get("share_count")),
        "share_url": f"https://www.douyin.com/video/{info.get('aweme_id', '')}",
    }


def _recorded_json(session):
    for resp in session.network.search(resource_type="xhr", limit=None, include_static=False):
        if resp.status != 200:
            continue
        try:
            data = resp.json()
        except Exception:
            continue
        if isinstance(data, dict):
            yield resp.url, data
    for resp in session.network.search(resource_type="fetch", limit=None, include_static=False):
        if resp.status != 200:
            continue
        try:
            data = resp.json()
        except Exception:
            continue
        if isinstance(data, dict):
            yield resp.url, data


def _parse_videos(session, keyword, page_text):
    kw_enc = urllib.parse.quote(keyword)
    api_items = []
    for url, data in _recorded_json(session):
        if not SEARCH_API_RE.search(url):
            continue
        if kw_enc not in url and keyword not in urllib.parse.unquote(url):
            continue
        items = data.get("data") or []
        if isinstance(items, list) and items:
            api_items.extend(items)
    if not api_items:
        all_search = [d for u, d in _recorded_json(session) if SEARCH_API_RE.search(u)]
        if len({id(d) for d in all_search}) == 1 and isinstance((all_search[0] or {}).get("data"), list):
            api_items = all_search[0]["data"]

    seen, rows = set(), []
    for item in list(api_items) + _rsc_aweme_objects(page_text):
        if not isinstance(item, dict):
            continue
        row = _extract_video(item, keyword, len(rows) + 1)
        if row is None or row["aweme_id"] in seen:
            continue
        seen.add(row["aweme_id"])
        rows.append(row)
        if len(rows) >= 10:
            break
    return rows


raw_cookie = _load_cookie_from_env()
cookies = _parse_cookies(raw_cookie)
if not raw_cookie:
    print("警告：.env 没有 DOUYIN_HOT_COOKIE，搜索页可能被验证码拦截")
elif not any("sessionid" in c["name"].lower() for c in cookies):
    print("警告：Cookie 里没有 sessionid 登录态，搜索页大概率仍会被验证码拦截")

cookie_injected = False

async def _setup_page(page):
    global cookie_injected
    if cookies and not cookie_injected:
        try:
            await page.context.add_cookies(cookies)
            cookie_injected = True
        except Exception as e:
            print(f"Cookie 注入失败: {e}")


async def _collect_videos():
    async with AsyncStealthySession(
        headless=True,
        network_idle=True,
        record_requests=True,
        max_pages=4,
        timeout=60000,
        wait=1500,
        google_search=False,
        page_setup=_setup_page,
    ) as session:
        all_videos = []
        captcha_warned = False
        for hot in rows:
            keyword = hot["title"]
            url = SEARCH_URL.format(kw=urllib.parse.quote(keyword, safe=""))
            page = await session.fetch(url)
            text = page.body.decode("utf-8", errors="ignore")
            if CAPTCHA_TEXT in text:
                if not captcha_warned:
                    print("搜索页出现验证码中间页：需要带 sessionid 的登录 Cookie 才能继续")
                    captcha_warned = True
                continue
            videos = _parse_videos(session, keyword, text)
            for video in videos:
                video["hot_rank"] = hot["rank"]
                all_videos.append(video)
            print(f"「{keyword}」: {len(videos)} 条视频")
        return all_videos


video_rows = await _collect_videos()
VIDEO_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
with VIDEO_OUTPUT_PATH.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=VIDEO_FIELDS)
    writer.writeheader()
    writer.writerows(video_rows)

print(f"已写入：{VIDEO_OUTPUT_PATH}，共 {len(video_rows)} 条视频")
missing = [row["title"] for row in rows if not any(v["keyword"] == row["title"] for v in video_rows)]
if missing:
    print(f"有 {len(missing)} 个关键词没有解析到视频，例如：{missing[:3]}")

警告：.env 没有 DOUYIN_HOT_COOKIE，搜索页可能被验证码拦截
